# Healthcare Popular Times Proxy Busyness ML

本 notebook 只负责运行 pipeline、展示结果、输出图表和说明。数据构建逻辑已经拆到：

```text
Data+ML/test/6.28-7.3/src/ml_feature_pipeline.py
```

目标仍是预测 `Google Popular Times proxy busyness`，不是实测真实客流。

当前特征输入优先来自 DB 规范化表（`venues`、`healthcare_profiles`、`pedestrian_ramps`），`venue_label_status_coverage_view.csv` 只保留样本锚点和覆盖统计。


## 0. Notebook 职责

- 运行离线 pipeline 生成/刷新 CSV 输出。
- 展示标签覆盖、特征覆盖、外部数据源审计、训练表规模。
- 展示输入指标与输出指标说明。
- 展示 DB 直取字段、外部静态 enrich 字段和 label-only 字段的来源归类。
- 不调用 SerpAPI live API，不写 DB，不执行 schema migration。
- 保留少量图表，方便汇报和检查。


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'Data+ML').exists() and (candidate / 'docs').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
PIPELINE_SRC = PROJECT_ROOT / 'Data+ML/test/6.28-7.3/src'
OUTPUT_DIR = PROJECT_ROOT / 'Data+ML/test/6.28-7.3/output'

sys.path.insert(0, str(PIPELINE_SRC))
from ml_feature_pipeline import run_pipeline

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PIPELINE_SRC exists:', PIPELINE_SRC.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)


## 1. 运行 Pipeline

该步骤会读取本地静态数据、DB 导出的规范化表和 SerpAPI cache，并刷新 `output/` 下的 CSV。默认不调用外部 API。


In [ ]:
outputs = run_pipeline(PROJECT_ROOT)
manifest = pd.read_csv(outputs['manifest'])
manifest


## 2.1 实现状态日志

这个块只根据当前 pipeline 输出判断状态，用来区分已落地、部分落地和仍缺失的能力。


In [ ]:
status_items = [
    {'area': 'feature_engineering', 'item': 'poi_density_300m from DB pedestrian_ramps', 'status': 'implemented' if (OUTPUT_DIR / 'spatial_features_v1.csv').exists() else 'missing'},
    {'area': 'feature_engineering', 'item': 'hospital_level / capacity', 'status': 'implemented' if (OUTPUT_DIR / 'healthcare_capacity_level_features_v1.csv').exists() else 'missing'},
    {'area': 'feature_engineering', 'item': 'is_business_hours', 'status': 'implemented' if (OUTPUT_DIR / 'ml_training_frame_v1.csv').exists() else 'missing'},
    {'area': 'data_lineage', 'item': 'DB feature source audit', 'status': 'implemented' if (OUTPUT_DIR / 'db_feature_source_audit.csv').exists() else 'missing'},
    {'area': 'model_training', 'item': 'Ridge / RandomForest / GradientBoosting', 'status': 'implemented' if (OUTPUT_DIR / 'model_metrics_v1.csv').exists() else 'missing'},
    {'area': 'evaluation', 'item': 'MAE / RMSE / R2 / F1 / recall', 'status': 'implemented' if (OUTPUT_DIR / 'model_metrics_v1.csv').exists() else 'missing'},
    {'area': 'ablation', 'item': 'feature ablation', 'status': 'implemented' if (OUTPUT_DIR / 'ablation_summary_v1.csv').exists() else 'missing'},
    {'area': 'serving', 'item': 'no_data fallback for out-of-scope venues', 'status': 'implemented'},
]
gap_log = pd.DataFrame(status_items)
gap_log['label'] = gap_log['status'].map({'implemented': '已实现', 'partial': '部分实现', 'missing': '未实现'})
display(gap_log)
display(gap_log.groupby(['area', 'status']).size().reset_index(name='count'))


## 2. 读取输出结果


In [ ]:
def read_output(name: str) -> pd.DataFrame:
    return pd.read_csv(OUTPUT_DIR / name, low_memory=False)

coverage_summary = read_output('coverage_summary.csv')
label_status = read_output('label_status_breakdown.csv')
feature_registry = read_output('feature_registry.csv')
popular_times_summary = read_output('popular_times_summary.csv')
training_summary = read_output('training_frame_summary.csv')
feature_coverage = read_output('feature_coverage_summary.csv')
spatial_audit = read_output('spatial_features_v1_audit.csv')
source_audit = read_output('healthcare_external_source_audit_v1.csv')
db_feature_source_audit = read_output('db_feature_source_audit.csv')
capacity_match_audit = read_output('healthcare_external_match_audit_v1.csv')
io_dictionary = read_output('input_output_field_dictionary.csv')
training_frame = read_output('ml_training_frame_v1.csv')
seasonal_baseline = read_output('seasonal_baseline.csv')
model_metrics = read_output('model_metrics_v1.csv')
model_predictions = read_output('model_test_predictions_v1.csv')
prediction_curve = read_output('prediction_curve_v1.csv')
ablation_summary = read_output('ablation_summary_v1.csv')

print('loaded outputs from', OUTPUT_DIR)


## 3. 标签覆盖与训练规模


In [ ]:
display(coverage_summary)
display(popular_times_summary)
display(training_summary)
if {'metric', 'value'}.issubset(coverage_summary.columns):
    healthcare_positive = int(coverage_summary.loc[coverage_summary['metric'].eq('healthcare_has_popular_times'), 'value'].iloc[0])
else:
    healthcare_positive = int(coverage_summary['has_popular_times'].iloc[0])

if {'metric', 'value'}.issubset(popular_times_summary.columns):
    unique_groups = int(popular_times_summary.loc[popular_times_summary['metric'].eq('unique_prediction_groups'), 'value'].iloc[0])
else:
    unique_groups = int(popular_times_summary['unique_prediction_groups'].iloc[0])
print(f"{healthcare_positive}: healthcare venues labeled as has_popular_times")
print(f"{unique_groups}: unique SerpAPI place IDs with parseable hourly popular-times data")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

healthcare_counts = coverage_summary.iloc[0][['has_popular_times', 'no_popular_times', 'search_not_matched']]
healthcare_counts.plot(kind='bar', ax=axes[0], color=['#2E7D32', '#F9A825', '#607D8B'])
axes[0].set_title('Healthcare Label Coverage')
axes[0].set_ylabel('Venues')
axes[0].tick_params(axis='x', rotation=25)

split_counts = training_frame['split'].value_counts().reindex(['train', 'val', 'test']).fillna(0)
split_counts.plot(kind='bar', ax=axes[1], color=['#1565C0', '#6A1B9A', '#C62828'])
axes[1].set_title('Group-aware Split Rows')
axes[1].set_ylabel('Hourly rows')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## 4. 特征登记与覆盖率


In [ ]:
display(feature_registry)

display(feature_coverage.sort_values('coverage_pct', ascending=False))


In [ ]:
plot_df = feature_coverage.sort_values('coverage_pct', ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df['feature'], plot_df['coverage_pct'], color='#006D77')
ax.set_xlabel('Non-null coverage (%)')
ax.set_title('Feature Coverage in Training Frame')
ax.set_xlim(0, 105)
for idx, value in enumerate(plot_df['coverage_pct']):
    ax.text(value + 1, idx, f'{value:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.show()


## 5. 外部数据源与匹配审计

这里检查 MTA / Citi Bike / DB ramps / NYS / CMS 静态文件是否被成功读取，以及 hospital/capacity 匹配状态。


In [ ]:
display(spatial_audit)
display(source_audit)
display(db_feature_source_audit)

match_status = capacity_match_audit['match_status'].value_counts(dropna=False).rename_axis('match_status').reset_index(name='venues')
cms_status = capacity_match_audit['cms_match_status'].value_counts(dropna=False).rename_axis('cms_match_status').reset_index(name='venues')
display(match_status)
display(cms_status)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

match_status.set_index('match_status')['venues'].plot(kind='bar', ax=axes[0], color='#455A64')
axes[0].set_title('NYS Facility Match Status')
axes[0].set_ylabel('Venues')
axes[0].tick_params(axis='x', rotation=35)

cms_status.set_index('cms_match_status')['venues'].plot(kind='bar', ax=axes[1], color='#8D6E63')
axes[1].set_title('CMS Match Status')
axes[1].set_ylabel('Venues')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


## 5.1 Urban Activity Spatial Features (v1)

These are spatial urban activity proxy features, not live traffic observations.

这些是城市活动空间代理特征，不是实时交通观测。

v1 接入 `6.15-6.20/output/venue_coverage_detail.csv`，使用 Citi Bike / MTA / Traffic 的最近距离和覆盖标记，构建综合空间活动 proxy score。

```text
urban_activity_spatial_score = 0.4 * citibike_score + 0.4 * mta_score + 0.2 * traffic_score
score = max(0, 100 * (1 - distance_m / 500))
缺失距离记为 0 分
```

v2 暂不实现：`mta_hourly_ridership` / `citibike_station_activity` / `nyc_traffic_hourly_volume` / `urban_activity_proxy_score`。

In [ ]:
urban_activity = read_output('urban_activity_spatial_features_v1.csv')
urban_activity_audit = read_output('urban_activity_spatial_features_v1_audit.csv')

display(urban_activity_audit)
print(f"Urban activity features rows: {len(urban_activity)}")
ua_cols = [
    'citibike_nearest_distance_m', 'mta_nearest_distance_m', 'traffic_nearest_distance_m',
    'citibike_covered_200m', 'mta_covered_200m', 'traffic_covered_500m',
    'urban_activity_spatial_score',
]
existing_ua = [c for c in ua_cols if c in urban_activity.columns]
print("\nNon-null coverage:")
print(urban_activity[existing_ua].notna().mean().round(3))
print("\nurban_activity_spatial_score summary:")
print(urban_activity['urban_activity_spatial_score'].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
dist_cols = ['citibike_nearest_distance_m', 'mta_nearest_distance_m', 'traffic_nearest_distance_m']
titles = ['Citi Bike Nearest Distance (m)', 'MTA Nearest Distance (m)', 'Traffic Nearest Distance (m)']
for ax, col, title in zip(axes, dist_cols, titles):
    data = urban_activity[col].dropna()
    if len(data):
        ax.hist(data.clip(upper=1000), bins=40, color='#006D77', edgecolor='white')
        ax.set_title(title)
        ax.set_xlabel('Distance (m)')
        ax.set_ylabel('Venues')
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

score_data = urban_activity['urban_activity_spatial_score'].dropna()
if len(score_data):
    axes[0].hist(score_data, bins=40, color='#2E86AB', edgecolor='white')
    axes[0].set_title('Urban Activity Spatial Score Distribution')
    axes[0].set_xlabel('Score')
    axes[0].set_ylabel('Venues')
    axes[0].axvline(score_data.mean(), color='red', linestyle='--', label=f'mean={score_data.mean():.1f}')
    axes[0].legend()

    coverage_bools = urban_activity[['citibike_covered_200m', 'mta_covered_200m', 'traffic_covered_500m']].apply(pd.to_numeric, errors='coerce').mean() * 100
    coverage_bools.plot(kind='bar', ax=axes[1], color=['#2E7D32', '#1565C0', '#F9A825'])
    axes[1].set_title('Coverage Rate (%)')
    axes[1].set_ylabel('Percent')
    axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

## 6. 输入指标与输出指标说明

`role = input_feature` / `input_feature_optional` 是模型输入候选；`target` 是训练标签；`model_output` 是后续前端或模型服务应输出的字段。


In [ ]:
display(io_dictionary)
display(io_dictionary.groupby('source_group').size().reset_index(name='fields'))


## 7. 训练表与 Baseline 输出样例


In [ ]:
display(training_frame.head(10))
display(seasonal_baseline.head(10))


## 8. 模型训练、评估与预测曲线

这里展示脚本产出的回归指标、测试集预测样例和 12h 预测曲线。


In [ ]:
display(model_metrics)
display(model_predictions.head(10))
display(prediction_curve)


In [ ]:
test_metrics = model_metrics[model_metrics['split'].eq('test')].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
test_metrics.set_index('model_name')['mae'].plot(kind='bar', ax=axes[0], color='#2E86AB')
axes[0].set_title('Test MAE by Model')
axes[0].set_ylabel('MAE')
test_metrics.set_index('model_name')['macro_f1'].plot(kind='bar', ax=axes[1], color='#7D3C98')
axes[1].set_title('Test Macro F1 by Model')
axes[1].set_ylabel('Macro F1')
plt.tight_layout()
plt.show()


## 9. 消融结果

这里展示 baseline、空间特征和容量特征的 ablation；缺失源会在表内标记为 `missing_source`。


In [ ]:
display(ablation_summary)


In [ ]:
ablation_ok = ablation_summary[ablation_summary['status'].eq('ok')].copy()
if not ablation_ok.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(ablation_ok['block_name'], ablation_ok['mae'], color='#006D77')
    ax.set_ylabel('MAE')
    ax.set_title('Ablation MAE on Test Split')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()


### 9.1 Ablation: Urban Activity Spatial vs Baseline

对比 baseline / mobility / urban_activity_spatial / full_available 四组消融结果。

In [ ]:
compare_blocks = ['baseline', 'mobility', 'urban_activity_spatial', 'full_available']
ablation_compare = ablation_summary[ablation_summary['block_name'].isin(compare_blocks)].copy()
display(ablation_compare[['block_name', 'status', 'model_name', 'feature_count', 'mae', 'rmse', 'r2', 'macro_f1']])

ok_compare = ablation_compare[ablation_compare['status'].eq('ok')]
if not ok_compare.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ok_compare.set_index('block_name')['mae'].plot(kind='bar', ax=axes[0], color='#006D77')
    axes[0].set_title('Ablation MAE: Urban Activity vs Baseline')
    axes[0].set_ylabel('MAE')
    axes[0].tick_params(axis='x', rotation=25)
    ok_compare.set_index('block_name')['macro_f1'].plot(kind='bar', ax=axes[1], color='#7D3C98')
    axes[1].set_title('Ablation Macro F1: Urban Activity vs Baseline')
    axes[1].set_ylabel('Macro F1')
    axes[1].tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()

## 10. 输出文件清单

这些文件是当前 notebook 的主要产物，可供后续模型训练、前端样例和汇报引用。


In [ ]:
manifest


## 11. 仍需注意

- `busyness_score` 是 Google Popular Times weak label，不是真实 foot traffic。
- split 必须按 `prediction_group_id` 分组，禁止 hourly rows 随机切分。
- `capacity` / `cms_rating` 覆盖率低是正常现象，因为并非所有 healthcare venue 都是医院。
- `is_business_hours` 已从 SerpAPI per-day hours 解析；第一版只用于 serving/post-processing 截断，非营业时间输出 `no_data`。
